# LeetCode #1001: Grid Illumination

https://leetcode.com/problems/grid-illumination/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O((n^2 + q) \cdot n^2)$ | $O(n^2)$ |
| **Optimal: Hash Map Counters ★** | $O(n + q)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Store all lamp positions. For each query, scan every lamp to see if it illuminates the query cell (same row, column, or diagonal). For each lamp to turn off, rescan to find lamps in its 3×3 neighbourhood. Hopelessly slow for $n \approx 10^9$.

### Optimal: Hash Map Counters ★
Maintain four hash maps counting how many lamps illuminate each row, column, main diagonal ($r - c$), and anti-diagonal ($r + c$). A cell is lit if any of its four counters is positive. To turn off a lamp, decrement its four counters and remove the lamp.

**Constraints:**
* $1 \le n \le 10^9$
* $0 \le lamps.length \le 20000$
* $0 \le queries.length \le 20000$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    public int[] GridIllumination(int n, int[][] lamps, int[][] queries) {
        // Track active lamps in a set to avoid double-counting duplicates
        var lampSet = new HashSet<long>();
        var row = new Dictionary<int, int>();
        var col = new Dictionary<int, int>();
        var diag = new Dictionary<int, int>();   // r - c
        var anti = new Dictionary<int, int>();   // r + c

        foreach (var lamp in lamps) {
            long key = (long)lamp[0] * 1_000_000_007 + lamp[1];
            if (!lampSet.Add(key)) continue;   // duplicate — ignore
            Increment(row, lamp[0]);
            Increment(col, lamp[1]);
            Increment(diag, lamp[0] - lamp[1]);
            Increment(anti, lamp[0] + lamp[1]);
        }

        int[] ans = new int[queries.Length];
        int[] dr = {-1,-1,-1,0,0,0,1,1,1};
        int[] dc = {-1,0,1,-1,0,1,-1,0,1};

        for (int q = 0; q < queries.Length; q++) {
            int r = queries[q][0], c = queries[q][1];
            // Cell is lit if any counter covering it is positive
            if (Get(row, r) > 0 || Get(col, c) > 0 || Get(diag, r-c) > 0 || Get(anti, r+c) > 0)
                ans[q] = 1;

            // Turn off every lamp in the 3x3 neighbourhood
            for (int d = 0; d < 9; d++) {
                int nr = r + dr[d], nc = c + dc[d];
                long key = (long)nr * 1_000_000_007 + nc;
                if (lampSet.Remove(key)) {
                    Decrement(row, nr);
                    Decrement(col, nc);
                    Decrement(diag, nr - nc);
                    Decrement(anti, nr + nc);
                }
            }
        }
        return ans;
    }

    void Increment(Dictionary<int,int> d, int k) {
        d.TryGetValue(k, out int v); d[k] = v + 1;
    }
    void Decrement(Dictionary<int,int> d, int k) {
        if (d.TryGetValue(k, out int v) && v > 1) d[k] = v - 1; else d.Remove(k);
    }
    int Get(Dictionary<int,int> d, int k) {
        d.TryGetValue(k, out int v); return v;
    }
}

### Python

In [ ]:
from collections import defaultdict

class Solution:
    def gridIllumination(self, n: int, lamps: list[list[int]], queries: list[list[int]]) -> list[int]:
        lamp_set = set()
        row = defaultdict(int)
        col = defaultdict(int)
        diag = defaultdict(int)   # r - c
        anti = defaultdict(int)   # r + c

        for r, c in lamps:
            if (r, c) in lamp_set:
                continue  # skip duplicates to avoid over-counting
            lamp_set.add((r, c))
            row[r] += 1
            col[c] += 1
            diag[r - c] += 1
            anti[r + c] += 1

        ans = []
        dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,0),(0,1),(1,-1),(1,0),(1,1)]

        for qr, qc in queries:
            # Lit if any counter for this cell's row/col/diag is positive
            lit = row[qr] or col[qc] or diag[qr - qc] or anti[qr + qc]
            ans.append(1 if lit else 0)

            # Extinguish lamps in the 3x3 neighbourhood
            for dr, dc in dirs:
                nr, nc = qr + dr, qc + dc
                if (nr, nc) in lamp_set:
                    lamp_set.remove((nr, nc))
                    row[nr] -= 1
                    col[nc] -= 1
                    diag[nr - nc] -= 1
                    anti[nr + nc] -= 1

        return ans

### Go

In [ ]:
func gridIllumination(n int, lamps [][]int, queries [][]int) []int {
	type point struct{ r, c int }
	lampSet := map[point]bool{}
	rowCnt := map[int]int{}
	colCnt := map[int]int{}
	diagCnt := map[int]int{} // r - c
	antiCnt := map[int]int{} // r + c

	inc := func(m map[int]int, k int) { m[k]++ }
	dec := func(m map[int]int, k int) {
		if m[k] > 1 { m[k]-- } else { delete(m, k) }
	}

	for _, l := range lamps {
		p := point{l[0], l[1]}
		if lampSet[p] { continue } // deduplicate
		lampSet[p] = true
		inc(rowCnt, l[0]); inc(colCnt, l[1])
		inc(diagCnt, l[0]-l[1]); inc(antiCnt, l[0]+l[1])
	}

	dirs := [][2]int{{-1,-1},{-1,0},{-1,1},{0,-1},{0,0},{0,1},{1,-1},{1,0},{1,1}}
	ans := make([]int, len(queries))

	for q, qr := range queries {
		r, c := qr[0], qr[1]
		// Check if the queried cell is covered by any illuminated axis
		if rowCnt[r] > 0 || colCnt[c] > 0 || diagCnt[r-c] > 0 || antiCnt[r+c] > 0 {
			ans[q] = 1
		}
		// Turn off lamps in the 3x3 neighbourhood
		for _, d := range dirs {
			nr, nc := r+d[0], c+d[1]
			p := point{nr, nc}
			if lampSet[p] {
				delete(lampSet, p)
				dec(rowCnt, nr); dec(colCnt, nc)
				dec(diagCnt, nr-nc); dec(antiCnt, nr+nc)
			}
		}
	}
	return ans
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet};

impl Solution {
    pub fn grid_illumination(n: i32, lamps: Vec<Vec<i32>>, queries: Vec<Vec<i32>>) -> Vec<i32> {
        let mut lamp_set: HashSet<(i32,i32)> = HashSet::new();
        let mut row: HashMap<i32,i32> = HashMap::new();
        let mut col: HashMap<i32,i32> = HashMap::new();
        let mut diag: HashMap<i32,i32> = HashMap::new(); // r - c
        let mut anti: HashMap<i32,i32> = HashMap::new(); // r + c

        for l in &lamps {
            let (r, c) = (l[0], l[1]);
            if !lamp_set.insert((r, c)) { continue; } // deduplicate
            *row.entry(r).or_default() += 1;
            *col.entry(c).or_default() += 1;
            *diag.entry(r - c).or_default() += 1;
            *anti.entry(r + c).or_default() += 1;
        }

        let dirs = [(-1i32,-1i32),(-1,0),(-1,1),(0,-1),(0,0),(0,1),(1,-1),(1,0),(1,1)];
        let mut ans = Vec::with_capacity(queries.len());

        for q in &queries {
            let (r, c) = (q[0], q[1]);
            // Lit if any counter for this cell's row, col, or diagonal is positive
            let lit = *row.get(&r).unwrap_or(&0) > 0
                || *col.get(&c).unwrap_or(&0) > 0
                || *diag.get(&(r-c)).unwrap_or(&0) > 0
                || *anti.get(&(r+c)).unwrap_or(&0) > 0;
            ans.push(if lit { 1 } else { 0 });

            // Extinguish each lamp in the 3x3 neighbourhood
            for &(dr, dc) in &dirs {
                let (nr, nc) = (r + dr, c + dc);
                if lamp_set.remove(&(nr, nc)) {
                    Self::dec(&mut row, nr);
                    Self::dec(&mut col, nc);
                    Self::dec(&mut diag, nr - nc);
                    Self::dec(&mut anti, nr + nc);
                }
            }
        }
        ans
    }

    fn dec(m: &mut HashMap<i32,i32>, k: i32) {
        let v = m.entry(k).or_default();
        if *v > 1 { *v -= 1; } else { m.remove(&k); }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `n=5, lamps=[[0,0],[4,4]], queries=[[0,0],[4,4]]`
Both lamps illuminate their own cells; each query returns 1. After first query the lamp at (0,0) turns off, but the second lamp still illuminates (4,4).

### 2. Slightly Complex
**Input:** `n=5, lamps=[[0,0],[1,1]], queries=[[1,1]]`
Both lamps share the same main diagonal ($r - c = 0$), so (1,1) is doubly illuminated; the anti-diagonal counter for (1,1) is also positive. Result: 1. Both lamps are then extinguished because they lie in the 3×3 neighbourhood of the query.

### 3. Edge Case: Time Factor
**Input:** `lamps` contains 20 000 entries, `queries` contains 20 000 entries each targeting a fresh unlit cell.
Every query performs 9 neighbourhood lookups; total work is $O(n + q) = O(40000)$, well within time limits — demonstrating $O(1)$ per query.

### 4. Edge Case: Space Factor
**Input:** All 20 000 lamps are distinct and spread across different rows/columns/diagonals.
All four counters grow to $\approx 20000$ entries each, totalling $O(n)$ space. No lamp shares an axis, so no early counter pruning occurs.

### 5. Almost-Impossible but Plausible
**Input:** `n=10^9, lamps=[[0,0],[999999999,999999999]], queries=[[500000000,500000000]]`
The query cell is not on any axis of either lamp, so the answer is 0. The grid spans $10^{18}$ cells but the algorithm only touches the two lamp entries — demonstrating that grid size is irrelevant to runtime.